# Concatenate embedding chunks in memory

This notebook concatenates chunk `.h5ad` files generated by `embed_chunk.py` (which store embeddings in `obsm` and usually do **not** have `X`).

It also runs a sanity check that the total number of rows in concatenated embeddings matches the row count of the original input `.h5ad`.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import anndata as ad
import tqdm



## merge adata
path_1 = "/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/outputs/train_and_predict/real_embeddings_film_otfm_unipert/predictions_part1.h5ad"
path_2 = "/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/outputs/train_and_predict/real_embeddings_film_otfm_unipert/predictions_part2.h5ad"
out_path = "/lustre/groups/ml01/workspace/xiaotong.fu/pancellflow/outputs/train_and_predict/real_embeddings_film_otfm_unipert/predictions.h5ad"
adata_1 = sc.read_h5ad(path_1)
adata_2 = sc.read_h5ad(path_2)
adata = adata_1.concat(adata_2)
adata.write_h5ad(out_path)

INPUT_H5AD = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe.h5ad")
CHUNKS_DIR = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/test")
OUT_H5AD = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/tahoe_w_se.h5ad")
OBSM_KEY = "X_state"

assert INPUT_H5AD.exists(), f"Missing input: {INPUT_H5AD}"
assert CHUNKS_DIR.exists(), f"Missing chunks dir: {CHUNKS_DIR}"

In [4]:
def chunk_sort_key(path: Path):
    # Natural-ish sort: if filename has integers, sort by those; fallback to name.
    nums = re.findall(r"\\d+", path.stem)
    if nums:
        return tuple(int(x) for x in nums)
    return (path.name,)

chunk_files = sorted(CHUNKS_DIR.glob("*.h5ad"), key=chunk_sort_key)
print(f"Found {len(chunk_files)} chunk files in {CHUNKS_DIR}")
assert len(chunk_files) > 0, "No chunk .h5ad files found."

embeddings = []
rows = []
dims = []
for fp in tqdm.tqdm(chunk_files):
    a = ad.read_h5ad(fp)
    assert OBSM_KEY in a.obsm, f"{fp} does not have obsm['{OBSM_KEY}']"
    emb = np.asarray(a.obsm[OBSM_KEY])
    embeddings.append(emb)
    rows.append(int(emb.shape[0]))
    dims.append(int(emb.shape[1]))

dim_set = sorted(set(dims))
assert len(dim_set) == 1, f"Embedding dimensions are inconsistent across chunks: {dim_set}"

report = pd.DataFrame({
    "file": [p.name for p in chunk_files],
    "n_rows": rows,
    "n_dim": dims,
})
display(report.head())
print(f"Total rows across chunks: {sum(rows):,}")
print(f"Embedding dim: {dim_set[0]}")

Found 200 chunk files in /lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/test


100%|██████████| 200/200 [1:16:35<00:00, 22.98s/it]


,file,n_rows,n_dim
0,chunk_00000.h5ad,447117,2058
1,chunk_00001.h5ad,447117,2058
2,chunk_00002.h5ad,447117,2058
3,chunk_00003.h5ad,447117,2058
4,chunk_00004.h5ad,447117,2058


Total rows across chunks: 89,423,257
Embedding dim: 2058


In [5]:
# Read input metadata (obs) without materializing X in memory.
adata_in = ad.read_h5ad(INPUT_H5AD, backed="r")
n_input = int(adata_in.n_obs)
obs_in = adata_in.obs.copy()

concat_emb = np.concatenate(embeddings, axis=0).astype(np.float32, copy=False)
n_concat = int(concat_emb.shape[0])

print(f"Input n_obs:   {n_input:,}")
print(f"Concat n_rows: {n_concat:,}")
assert n_concat == n_input, (
    f"Row mismatch: concatenated embeddings ({n_concat}) != input n_obs ({n_input})."
)

# Build output h5ad without X, preserving original obs rows.
adata_out = ad.AnnData(obs=obs_in)
adata_out.obsm[OBSM_KEY] = concat_emb
adata_out.uns["concat_info"] = {
    "source_input_h5ad": str(INPUT_H5AD),
    "source_chunks_dir": str(CHUNKS_DIR),
    "n_chunks": len(chunk_files),
    "obsm_key": OBSM_KEY,
}

OUT_H5AD.parent.mkdir(parents=True, exist_ok=True)
adata_out.write_h5ad(OUT_H5AD, compression="gzip")
print(f"Wrote: {OUT_H5AD}")
print(adata_out)

: 

: 

: 